# E-Commerce Lakehouse: Silver Payments

## Standardize payment fields and data types

Preserve order identifiers, standardize payment type text, and
convert payment sequence and installment counts to integers.
Convert payment_value to a decimal with two decimal places.
Validate the results before saving.

In [ ]:
from pyspark.sql.functions import col, trim, lower

bronze_payments_df = spark.table(
    "workspace.ecommerce.bronze_payments"
)

silver_payments_df = (
    bronze_payments_df
    .withColumn("order_id", trim(col("order_id")))
    .withColumn("payment_type", lower(trim(col("payment_type"))))
    .withColumn(
        "payment_sequential",
        col("payment_sequential").cast("int")
    )
    .withColumn(
        "payment_installments",
        col("payment_installments").cast("int")
    )
    .withColumn(
        "payment_value",
        col("payment_value").cast("decimal(12,2)")
    )
)

silver_payments_df.printSchema()
display(silver_payments_df.limit(5))

## Validate Silver payments

Compare row counts and null counts before and after conversion.
Check required fields, payment key uniqueness, numeric values,
payment types, and links to Silver orders.

Inspect zero-value payments and zero installments without
automatically treating them as errors.

In [ ]:
from pyspark.sql import functions as F

converted_columns = [
    "payment_sequential",
    "payment_installments",
    "payment_value"
]

def null_summary(df):
    return df.agg(
        F.count("*").alias("rows"),
        *[
            F.count(
                F.when(F.col(name).isNull(), 1)
            ).alias(f"{name}_nulls")
            for name in converted_columns
        ]
    )

print("Bronze summary:")
null_summary(bronze_payments_df).show()

print("Silver summary:")
null_summary(silver_payments_df).show()

silver_payments_df.agg(
    F.count(F.when(
        F.col("order_id").isNull()
        | (F.trim(F.col("order_id")) == ""), 1
    )).alias("missing_order_ids"),

    F.count(F.when(
        F.col("payment_type").isNull()
        | (F.trim(F.col("payment_type")) == ""), 1
    )).alias("missing_payment_types"),

    F.count(F.when(
        F.col("payment_sequential") <= 0, 1
    )).alias("nonpositive_sequences"),

    F.count(F.when(
        F.col("payment_installments") < 0, 1
    )).alias("negative_installments"),

    F.count(F.when(
        F.col("payment_value") < 0, 1
    )).alias("negative_payment_values"),

    F.count(F.when(
        F.col("payment_installments") == 0, 1
    )).alias("zero_installments"),

    F.count(F.when(
        F.col("payment_value") == 0, 1
    )).alias("zero_payment_values")
).show(truncate=False)

duplicate_keys = (
    silver_payments_df
    .groupBy("order_id", "payment_sequential")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

print("Duplicate payment keys:", duplicate_keys)

silver_payments_df.groupBy("payment_type").count().show()

order_ids_df = spark.table(
    "workspace.ecommerce.silver_orders"
).select("order_id")

unmatched_count = silver_payments_df.join(
    order_ids_df,
    on="order_id",
    how="left_anti"
).count()

print("Payments without a matching order:", unmatched_count)

## Inspect unusual payment records

Inspect payments with zero installments, zero payment value,
or an undefined payment type. Preserve these records while
determining how they should be treated in analysis.

In [ ]:
unusual_payments_df = silver_payments_df.filter(
    (F.col("payment_installments") == 0)
    | (F.col("payment_value") == 0)
    | (F.col("payment_type") == "not_defined")
)

display(
    unusual_payments_df.select(
        "order_id",
        "payment_sequential",
        "payment_type",
        "payment_installments",
        "payment_value"
    ).orderBy("payment_type", "order_id", "payment_sequential")
)

In [ ]:
display(
    unusual_payments_df.select(
        "order_id",
        "payment_sequential",
        "payment_type",
        "payment_installments",
        "payment_value"
    ).orderBy("payment_type", "order_id", "payment_sequential")
)

## Flag unusual payments and refresh Silver

Preserve all payment records and flag zero installments, zero
payment values, and undefined payment types.

Validate payment keys and required numeric fields before replacing
the Silver table. Verify saved row and flag counts after writing.

In [ ]:
from pyspark.sql import functions as F

silver_payments_df = (
    silver_payments_df
    .withColumn(
        "zero_installments_flag",
        F.col("payment_installments") == 0
    )
    .withColumn(
        "zero_payment_value_flag",
        F.col("payment_value") == 0
    )
    .withColumn(
        "undefined_payment_type_flag",
        F.col("payment_type") == "not_defined"
    )
)

invalid_count = silver_payments_df.filter(
    F.col("order_id").isNull()
    | (F.trim(F.col("order_id")) == "")
    | F.col("payment_type").isNull()
    | (F.trim(F.col("payment_type")) == "")
    | F.col("payment_sequential").isNull()
    | (F.col("payment_sequential") <= 0)
    | F.col("payment_installments").isNull()
    | (F.col("payment_installments") < 0)
    | F.col("payment_value").isNull()
    | (F.col("payment_value") < 0)
).count()

duplicate_count = (
    silver_payments_df
    .groupBy("order_id", "payment_sequential")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

row_count = silver_payments_df.count()

if (
    row_count != 103886
    or invalid_count != 0
    or duplicate_count != 0
):
    raise ValueError(
        "Silver payment validation failed. "
        "Stopped before replacing the table."
    )

table_name = "workspace.ecommerce.silver_payments"

(
    silver_payments_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(table_name)
)

saved_payments_df = spark.table(table_name)

saved_checks = saved_payments_df.agg(
    F.count("*").alias("saved_rows"),
    F.count(F.when(
        F.col("zero_installments_flag"), 1
    )).alias("zero_installments"),
    F.count(F.when(
        F.col("zero_payment_value_flag"), 1
    )).alias("zero_payment_values"),
    F.count(F.when(
        F.col("undefined_payment_type_flag"), 1
    )).alias("undefined_payment_types")
).first()

if (
    saved_checks["saved_rows"] != row_count
    or saved_checks["zero_installments"] != 2
    or saved_checks["zero_payment_values"] != 9
    or saved_checks["undefined_payment_types"] != 3
):
    raise ValueError("Saved payment counts differ from the expected dataset.")

print("Silver payments refreshed:", saved_checks["saved_rows"])
print("Zero installments:", saved_checks["zero_installments"])
print("Zero payment values:", saved_checks["zero_payment_values"])
print("Undefined payment types:", saved_checks["undefined_payment_types"])